# 02 Spike Timing と Spiking Manifesto 型 LUT

## 1. 研究目的

Spiking Manifesto（arXiv:2512.11843）の中心的な計算原理を、小さな例で理解します。

- 発火**率**ではなく、スパイクの**相対的な発火時刻・発火順序**を情報として使う。
- 発火時刻の**大小比較（pairwise comparison）**からビット列を作り、それを**整数 index** にして、
  **Look-Up Table（LUT）の行を選ぶ**。行列の掛け算は使わない。

## 2. 理論的背景

### Latency vector

各ニューロンが観測時間内に 1 回だけ発火するとき、ニューロン $i$ の発火時刻 $x_i$ を並べたベクトル $x$ を **latency vector** と呼びます。

### 発火順序と $n!$

$n$ 個のニューロンが同時発火なしで 1 回ずつ発火すると、発火順序は $n!$ 通りあります。ただしこれは**理論上の順列数**で、ノイズのある実際のスパイクで安定して区別できる数は普通もっと少なくなります。

### Spiking Manifesto 型 LUT（論文 Section II、Fig. 6、式 (1)〜(3)）

1. 各 LUT に $n_c$ 組の **anchor pair** $(a, b)$ を割り当てる（論文では初期化時にランダム）
2. 各組について $x_a > x_b$ なら 1、そうでなければ 0（**pairwise comparison**）
3. $n_c$ 個のビットを 2 進数として読み、**index**（0〜$2^{n_c}-1$）にする（式 (1)）
4. その LUT の index 行目を選ぶ
5. $n_t$ 個の LUT で選ばれた行を**足し合わせて**出力 $y$ にする（式 (2)・(3)）

### 論文の LUT モデルと、この教材の簡略モデルの違い

| 項目 | 論文 | この教材（`spike_lut.py`） |
| --- | --- | --- |
| 出力の次元 | 入力と同じ（$y$ がそのまま次の latency vector） | 任意の $m$（単一 LUT の例は $m=2$） |
| ビットの並び | Fig. 6 の例は 1 つ目が上位ビット。著者コードは 1 つ目が最下位ビット | 1 つ目の比較を**最上位ビット** |
| 同時発火 | 比較規則として明記なし（式 (1) は $u>0$） | ビット **0** と定義 |
| latency の基準 | 区間の中心からの時刻（負もあり得る） | 観測区間の始まりからの時刻（0 以上） |
| 行の値 | 学習（surrogate gradient・誤差逆伝播）で決める | 手で決めた値、または乱数（seed 固定） |
| 規模 | 例：$n_t=64$、$n_c=10$、Transformer で約 8 億パラメータ | $n_t=3$、$n_c=3$ 程度 |

ビットの並べ方を変えても**行の並び順が入れ替わるだけ**で、表現力は変わりません。

## 3. 実験環境（最初に必ず実行）

このセルは、E3 リポジトリの `.py` ファイル（`ann.py`、`spike_lut.py` など）を import できるように、
**作業ディレクトリ（E3 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E3 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E3 フォルダを ZIP にして、Colab の左の「ファイル」に `/content` へアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/` フォルダ（教科書のコード）は**公開リポジトリには含めない**約束です。
> 方法A で `textbook/` がない場合、`spike_encoding.py`・`benchmark.py` の import が失敗します。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                           # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e3-ann-snn-comparison"   # リポジトリの中の E3 フォルダ（ルートが E3 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e3_root(p):
    """spike_lut.py と ann.py があるフォルダを E3 のルートとみなす"""
    return (Path(p) / "spike_lut.py").exists() and (Path(p) / "ann.py").exists()


def find_e3_root():
    # 1) ローカル：notebooks/ の中、または E3 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e3_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e3_upload")
    for hit in glob.glob("/content/e3_upload/**/spike_lut.py", recursive=True):
        if is_e3_root(Path(hit).parent):
            return Path(hit).parent
    return None


E3_ROOT = find_e3_root()
if E3_ROOT is None:
    raise RuntimeError("E3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E3_ROOT)
if str(E3_ROOT) not in sys.path:
    sys.path.insert(0, str(E3_ROOT))
print("E3_ROOT:", E3_ROOT)
if not (E3_ROOT / "textbook" / "insertion_sort.py").exists():
    print("⚠️ textbook/ がありません。spike_encoding.py などの import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は役割ごとに固定する（ANN / LIF・SNN / LUT、CPU / GPU / TPU）
COLORS = {"ANN": "#2a78d6", "LIF": "#1baf7a", "LUT": "#eb6834",
          "CPU": "#2a78d6", "GPU": "#eb6834", "TPU": "#1baf7a", "neutral": "#8a8985"}

## 4. 使用するコード

| ファイル | 使う関数 |
| --- | --- |
| `spike_encoding.py` | `spike_order`（教科書の単純挿入ソートで発火順序を計算）、`spike_counts`、`num_orders`、`format_order`、`quantize` |
| `spike_lut.py` | `compare_pairs`、`bits_to_index`、`index_to_bits`、`select_row`、`SpikeLUT`、`lut_forward`、`make_random_tables` |

In [ ]:
import json
import random
import numpy as np

from spike_encoding import spike_order, spike_counts, num_orders, format_order, quantize
from spike_lut import (SpikeLUT, compare_pairs, bits_to_index, index_to_bits,
                       select_row, lut_forward, make_random_tables)

with open("data/sample_patterns.json", encoding="utf-8") as f:
    data = json.load(f)
window = data["window"]   # 観測時間 [0, window)

## 5. 実行例

### Part A：Spike timing ―― 発火数は同じ、順序は逆

In [ ]:
pattern_a = [0.1, 0.2, 0.3, 0.4]
pattern_b = [0.4, 0.3, 0.2, 0.1]

for name, p in [("pattern_a", pattern_a), ("pattern_b", pattern_b)]:
    print(f"{name}: latencies={p}  counts={spike_counts(p, window)}  order={spike_order(p)}")

assert spike_counts(pattern_a, window) == spike_counts(pattern_b, window)   # 発火数は同じ
assert spike_order(pattern_a) == [0, 1, 2, 3] and spike_order(pattern_b) == [3, 2, 1, 0]

### 6-1. 可視化：Spike raster plot

横軸が時刻、縦軸がニューロン番号です。どちらも各ニューロンが 1 回ずつ発火しています（発火率は同じ）が、**発火の順番が逆**です。

In [ ]:
def raster(ax, latencies, title, color=COLORS["LIF"]):
    for i, t in enumerate(latencies):
        ax.vlines(t, i - 0.35, i + 0.35, color=color, lw=3)
    ax.set_yticks(range(len(latencies)))
    ax.set_yticklabels([f"N{i}" for i in range(len(latencies))])
    ax.set_xlim(0, window); ax.set_xlabel("time"); ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), sharey=True)
raster(axes[0], pattern_a, "pattern A: order 0->1->2->3")
raster(axes[1], pattern_b, "pattern B: order 3->2->1->0", color=COLORS["LUT"])
axes[0].set_ylabel("neuron")
plt.tight_layout(); plt.show()

### Part B：Spike order と $n!$

`spike_order` は (発火時刻, 番号) の組を、教科書 第6章の**単純挿入ソート**で昇順に並べます。
同時発火のときは番号の小さい方が先になります。

In [ ]:
latencies = data["latency_example"]          # [0.3, 0.1, 0.4, 0.2]
order = spike_order(latencies)
print("latencies         :", latencies)
print("発火時刻の並べ替え:", sorted(latencies))
print("発火順序          :", order, "=", format_order(order))
print("pattern_a と pattern_b の順序は同じ？", spike_order(pattern_a) == spike_order(pattern_b))
assert order == [1, 3, 0, 2]

print()
for n in [3, 4, 5, 6, 10]:
    print(f"n = {n:2d}: n! = {num_orders(n):>12,}")

In [ ]:
# 6-2. 可視化：発火順序（何番目に発火したか）と n! の増え方
rank = [order.index(i) + 1 for i in range(len(latencies))]   # ニューロン i が何番目に発火したか
ns = list(range(1, 21))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2))
ax1.bar(range(len(rank)), rank, color=COLORS["LIF"])
ax1.set_xticks(range(len(rank))); ax1.set_xticklabels([f"N{i}" for i in range(len(rank))])
ax1.set_ylabel("firing rank (1 = first)"); ax1.set_title(f"spike order of {latencies}")
ax2.semilogy(ns, [num_orders(n) for n in ns], marker="o", ms=4, color=COLORS["LUT"], label="n!")
ax2.semilogy(ns, [2 ** n for n in ns], ls="--", color=COLORS["neutral"], label="2^n")
ax2.set_xlabel("number of neurons n"); ax2.set_ylabel("count (log)")
ax2.set_title("theoretical number of orders"); ax2.legend()
plt.tight_layout(); plt.show()

### Part C：Pairwise comparison

anchor pair $(a, b)$ ごとに $x_a > x_b$ を調べます。latency が大きい＝**後に**発火したので、ビット 1 は「$a$ が $b$ より後に発火した」ことを表します。

In [ ]:
anchor_pairs = [(0, 1), (1, 2), (2, 3)]
bits = compare_pairs(latencies, anchor_pairs)
for (a, b_), bit in zip(anchor_pairs, bits):
    print(f"x[{a}]={latencies[a]} > x[{b_}]={latencies[b_]} ? -> {bit}")
assert bits == [1, 0, 1]

### Part D：Binary index

1 つ目の比較を**最上位ビット**とする 2 進数として読みます：`[1, 0, 1]` → `101` → **5**。

In [ ]:
index = bits_to_index(bits)
print(f"bits {bits} -> binary {''.join(map(str, bits))} -> index {index}")
print("逆変換 index_to_bits:", index_to_bits(index, len(bits)))
assert index == 5 and index_to_bits(5, 3) == [1, 0, 1]

### Part E：LUT の行選択と出力 latency vector

単一の LUT（`data/sample_patterns.json`、$2^3=8$ 行、$m=2$）で行を選びます。続いて、乱数で作った複数の LUT（$n_t=3$、$n_c=3$、$m=4$、seed 2025）で、選ばれた行を足し合わせて**次の latency vector** を作ります。

In [ ]:
ex = data["lut_example"]
table = SpikeLUT(ex["n_inputs"], ex["anchor_pairs"], ex["rows"])
result = lut_forward(latencies, [table])
print("選ばれた index:", result["indices"][0], " 行:", select_row(table.rows, result["indices"][0]))
print("出力          :", result["output"])
assert result["indices"] == [5] and result["output"] == [0.4, 0.4]

# pattern A / B は発火数が同じでも、別の行が選ばれる
for name, p in [("pattern_a", pattern_a), ("pattern_b", pattern_b)]:
    r = lut_forward(p, [table])
    print(f"{name}: bits {r['bits'][0]} -> index {r['indices'][0]} -> output {r['output']}")

mex = data["multi_lut_example"]
tables = make_random_tables(mex["n_inputs"], mex["n_tables"], mex["n_comparisons"], mex["out_dim"], seed=mex["seed"])
multi = lut_forward(latencies, tables)
print()
for i, t in enumerate(tables):
    print(f"LUT {i}: anchors {t.anchors} bits {multi['bits'][i]} index {multi['indices'][i]} row {t.rows[multi['indices'][i]]}")
print("出力 latency vector y = Σ 選ばれた行:", [round(v, 2) for v in multi["output"]])
print("次の発火順序:", format_order(spike_order(multi["output"])))
assert [round(v, 2) for v in multi["output"]] == [0.47, 0.90, 0.77, 0.64]   # README 8章の値

### 6-3. 可視化：比較結果 → binary index → LUT の選択行 → 出力

左：pairwise comparison の結果（ビット）、中央：単一 LUT の全行（色の濃さが値、枠が選ばれた行）、右：複数 LUT の選択行と、その和である出力 latency vector。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 1.2, 1.6]})

ax = axes[0]
ax.bar([f"x{a}>x{b_}" for a, b_ in anchor_pairs], bits, color=COLORS["LUT"])
ax.set_ylim(0, 1.2); ax.set_yticks([0, 1]); ax.set_title(f"bits {bits} -> index {index}")

ax = axes[1]
rows = np.array(table.rows)
ax.imshow(rows, cmap="Oranges", aspect="auto", vmin=0)
ax.add_patch(plt.Rectangle((-0.5, index - 0.5), rows.shape[1], 1, fill=False, ec="black", lw=2))
ax.set_yticks(range(len(rows))); ax.set_yticklabels([f"{j} ({''.join(map(str, index_to_bits(j, 3)))})" for j in range(len(rows))])
ax.set_xticks(range(rows.shape[1])); ax.set_xlabel("output dim k"); ax.set_title("single LUT, selected row")
ax.grid(False)

ax = axes[2]
selected = np.array([t.rows[j] for t, j in zip(tables, multi["indices"])])
bottom = np.zeros(selected.shape[1])
shades = ["#f6c3a8", "#eb6834", "#a8401a"]
for i, row in enumerate(selected):
    ax.bar(range(len(row)), row, bottom=bottom, color=shades[i % 3], edgecolor="white", lw=1,
           label=f"LUT {i} row {multi['indices'][i]}")
    bottom += row
ax.set_xticks(range(len(bottom))); ax.set_xticklabels([f"y{k}" for k in range(len(bottom))])
ax.set_title("output latency vector y = sum of rows"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### ノイズに対する index の頑健性（例）

発火時刻にガウスノイズ（標準偏差 $\sigma$）を加え、LUT index が**元と同じまま**の割合を調べます（seed 固定）。
比較する 2 つの発火時刻が近いほど、小さなノイズで順序が入れ替わり、index が変わりやすくなります。

In [ ]:
def index_stability(base, tables, sigma, trials=500, seed=0):
    """ノイズを加えても、すべての LUT の index が元と同じだった割合"""
    rng = random.Random(seed)
    ref = lut_forward(base, tables)["indices"]
    same = 0
    for _ in range(trials):
        noisy = [t + rng.gauss(0, sigma) for t in base]
        same += lut_forward(noisy, tables)["indices"] == ref
    return same / trials

sigmas = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2]
stab_single = [index_stability(latencies, [table], s) for s in sigmas]
stab_multi = [index_stability(latencies, tables, s) for s in sigmas]

plt.figure()
plt.plot(sigmas, stab_single, marker="o", ms=6, color=COLORS["LUT"], label="1 LUT (nc=3)")
plt.plot(sigmas, stab_multi, marker="s", ms=6, color=COLORS["ANN"], label="3 LUTs (nc=3 each)")
plt.xlabel("timing noise sigma"); plt.ylabel("fraction with unchanged index")
plt.ylim(0, 1.05); plt.title(f"index stability, base = {latencies}"); plt.legend()
plt.show()

## 7. 結果の解釈

- pattern A と B は**発火数（発火率）が同じ**なので、発火率だけでは区別できません。しかし**発火順序**は逆で、LUT でも**違う行**が選ばれます。
- LUT の forward には**乗算がありません**。必要なのは比較、ビットから index への変換（シフトと加算）、行の読み出し、加算だけです。
- $n_c$ 回の比較で区別できるのは**高々 $2^{n_c}$ 通り**です。$n!$ 通りの発火順序をすべて区別できるわけではありません。比較どうしが依存していると、**到達できない index**（使われない行）も生じます。
- 近い latency vector は同じ index になりやすく、比較する 2 つの時刻が近いと index は変わりやすくなります。論文 Section VI は、これを **locality-sensitive hashing（LSH）** として説明しています。
- ここで使った行の値は**学習していない**手書き・乱数の値です。出力そのものに意味はなく、**計算の流れ**を確かめるための例です。

## 8. 学生向け TODO

- [ ] **TODO 1**：anchor pair の数を変更する（`my_anchors` に組を足す・減らす）。行数 $2^{n_c}$ も変わることに注意。
- [ ] **TODO 2**：spike timing（`my_latencies`）を変更する。
- [ ] **TODO 3**：LUT の行数を変更する（＝ $n_c$ を変える。`make_random_tables` の `n_comparisons`）。
- [ ] **TODO 4**：出力次元 `out_dim` を変更する（論文と同じ $m=n$ にして、出力を次の入力として繰り返し使ってみる）。
- [ ] **TODO 5**：`my_latencies[0]` を少しずつ動かして、LUT index が変わる境目を調べる。
- [ ] **TODO 6**：発火時刻にノイズを追加する（`index_stability` の `sigma`、`seed` を記録）。
- [ ] **TODO 7**：情報表現の頑健性を調べる（どんな入力・anchor pair だと index が変わりにくいか）。`quantize` で時刻を粗くした場合も調べる。

In [ ]:
my_latencies = [0.3, 0.1, 0.4, 0.2]            # TODO 2・5
my_anchors = [(0, 1), (1, 2), (2, 3)]          # TODO 1
print("bits :", compare_pairs(my_latencies, my_anchors))
print("index:", bits_to_index(compare_pairs(my_latencies, my_anchors)))

my_tables = make_random_tables(n_inputs=4, n_tables=3, n_comparisons=3, out_dim=4, seed=0)   # TODO 3・4
print("multi-LUT indices:", lut_forward(my_latencies, my_tables)["indices"])
# TODO 6・7：index_stability(my_latencies, my_tables, sigma) を sigma ごとに調べてグラフにしよう

## 9. 考察課題

1. 発火率表現と発火順序表現で、区別できるパターンの数（理論上）はどう違いますか。$2^n$ と $n!$ を比べてください。
2. 同時発火をビット 0 とする規則は、どのような場合に問題になりますか。`quantize` で分解能を粗くして確かめてください。
3. anchors を `[(0,1), (1,2), (0,2)]` にしたとき、$3! = 6$ 通りの発火順序から出る index を列挙し、$2^3=8$ 個のうち到達できない index を見つけてください。
4. 教科書のハッシュ法では衝突は「避けるもの」です。LUT（LSH 的な仕組み）では、似た入力が同じ index になること（衝突）にどんな意味がありますか。
5. この Notebook の LUT は**学習していません**。論文の surrogate gradient による学習（Section III）では、何を学習するのか調べてください。

## 10. 参考文献

- E. Izhikevich, "Spiking Manifesto", arXiv:2512.11843, 2025. <https://arxiv.org/abs/2512.11843>（arXiv の表示ライセンス：CC BY-NC-SA 4.0）
- 著者の公開コード：<https://github.com/izhikevich/SNN>（この教材には**コピーしていません**）
- W. Gerstner, W. M. Kistler, R. Naud, L. Paninski, *Neuronal Dynamics*, Cambridge University Press, 2014（LIF モデル）
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第5章 再帰、第6章 ソート）
- E3 リポジトリの `README.md`（特に 17章「論文との対応関係」）

> この Notebook は論文の中心的なアイデアを**教育用に簡略化**したものです。論文のモデル・著者の実装そのものではありません。